# Thermal Localized Power Viewer

Use the slider below to step through the coupled digital twin results and inspect the thermal power slice at a chosen `z` index.

The notebook expects `coupled_digital_twin_results.npz` to live in the same directory.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from ipywidgets import IntSlider, interact

from digital_twin import reshape_flux_vector

RESULTS_PATH = Path("coupled_digital_twin_results.npz")
if not RESULTS_PATH.exists():
    raise FileNotFoundError(f"Could not find {RESULTS_PATH}. Run digital_twin_driver.py first.")

results = np.load(RESULTS_PATH)
localized_power = results["localized_power"]
macro_time = results["time"]

print(f"Loaded {localized_power.shape[0]} macro frames from {RESULTS_PATH}")
print(f"Time range: {macro_time[0]:.3f} s to {macro_time[-1]:.3f} s")


def show_frame(frame_idx: int, z_index: int = 35) -> None:
    field = reshape_flux_vector(localized_power[int(frame_idx)])
    thermal_slice = field[z_index, :, :, 0]
    positive_slice = np.maximum(thermal_slice, 1e-12)
    positive_values = positive_slice[np.isfinite(positive_slice) & (positive_slice > 0)]
    vmin = float(np.min(positive_values)) if positive_values.size else 1e-12
    vmax = float(np.max(positive_slice))

    plt.figure(figsize=(7, 6))
    plt.imshow(
        positive_slice,
        origin="lower",
        cmap="inferno",
        norm=LogNorm(vmin=vmin, vmax=vmax),
    )
    plt.colorbar(label="Localized thermal power")
    plt.title(f"Thermal localized power | frame={frame_idx}, t={macro_time[frame_idx]:.3f} s, z={z_index}")
    plt.xlabel("X index")
    plt.ylabel("Y index")
    plt.tight_layout()
    plt.show()

In [ ]:
frame_slider = IntSlider(min=0, max=localized_power.shape[0] - 1, step=1, value=0, description="Frame")
z_slider = IntSlider(min=0, max=69, step=1, value=35, description="Z index")

interact(show_frame, frame_idx=frame_slider, z_index=z_slider)